# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [2]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "steven"

if not os.path.isdir("ECE1508_GenAI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    !cd ECE1508_GenAI && git pull

%cd ECE1508_GenAI

Cloning into 'ECE1508_GenAI'...
remote: Enumerating objects: 996, done.
remote: Counting objects: 100% (508/508), done.
remote: Compressing objects: 100% (384/384), done.
remote: Total 996 (delta 242), reused 382 (delta 124), pack-reused 488 (from 1)
Receiving objects: 100% (996/996), 37.33 MiB | 39.86 MiB/s, done.
Resolving deltas: 100% (443/443), done.
/content/ECE1508_GenAI


In [3]:
# torch is preinstalled on Colab; just need mplfinance + pyyaml
!pip install -q mplfinance pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.5 MB/s eta 0:00:00


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [5]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ECE1508_GenAI
plugins: anyio-4.14.2, typeguard-4.5.2, langsmith-0.10.2
collected 34 items                                                             

steven/tests/test_data_pipeline.py::test_reconstruct_prices_round_trip PASSED [  2%]
steven/tests/test_data_pipeline.py::test_anchor_correction_matches_close_0_for_all_horizon_bars PASSED [  5%]
steven/tests/test_data_pipeline.py::test_wick_components_non_negative PASSED [  8%]
steven/tests/test_data_pipeline.py::test_build_window_shapes_and_masks PASSED [ 11%]
steven/tests/test_data_pipeline.py::test_to_patchtst_input_patch_padding_mask PASSED [ 14%]
steven/tests/test_data_pipeline.py::test_window_sampler_unique_and_within_bounds PASSED [ 17%]
steven/tests/test_data_pipeline.py::test_window_sampler_respects_split_boundary PASSED [ 20%]

## Train PatchTST (real defaults: 20k windows/epoch, 20 epochs, configs/patchtst.yaml)

Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first instead of the full config.

In [ ]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst.yaml --device auto

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae.yaml)

In [ ]:
!python steven/src/train_cvae.py --config steven/configs/cvae.yaml --device auto

## Evaluate both models on the fixed test set

In [ ]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt \
  --device auto

## Refresh v1.md from this run

Rewrites the Results/backtest tables and sample images in `steven/v1.md` from the metrics.json + sample_plots this run just produced (see `steven/src/update_report.py`). Only the tables/images are rewritten -- surrounding prose (interpretation, caveats) is left as-is; review it by hand if the story changed. This only edits the file in the cloned repo here -- push/download separately if you want to keep it.

In [ ]:
!python steven/src/update_report.py

## Sync results back to GitHub

Commits `steven/outputs/` (checkpoints, metrics.json, sample_plots) and the regenerated `steven/v1.md` from this Colab runtime and pushes straight to the `steven` branch -- no manual zip/download step. That step wasn't reliably reaching the local machine: `files.download()`'s browser-download trick only works from the Colab web UI, not when this kernel is attached remotely (e.g. from VS Code's kernel picker), so nothing ever landed on disk.

Needs a GitHub personal access token with `repo` write scope for this push only -- entered via `getpass` below, never written to the notebook or committed anywhere.

In [ ]:
import getpass

token = getpass.getpass("GitHub PAT (repo write, used only for this push): ")

In [ ]:
%%bash -s "$token"
TOKEN="$1"
git config user.email "colab@ephemeral.local"
git config user.name "Colab Runtime"
git add steven/outputs steven/v1.md
if git diff --cached --quiet; then
  echo "Nothing to commit -- outputs/v1.md unchanged from last commit."
else
  git commit -m "Retrain + refresh results from Colab run"
  git push "https://${TOKEN}@github.com/WoodyChang21/ECE1508_GenAI.git" HEAD:steven
fi

### Fallback: zip + browser download

Only useful if you're running this notebook inside the actual Colab web UI (not a remote kernel) and would rather download a zip than push through git.

In [ ]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")